# 📓 Semantic Trace Search and Failure Clustering

The Records page searches traces by literal substring, but you usually know a
failure by *meaning*: "the assistant invented a citation", "the billing tool
timed out". And once you find one, the next question is whether it is an
isolated incident or a recurring failure mode.

This notebook builds both, locally, with standard text-analysis tools:
TF-IDF over unigrams and bigrams, a truncated SVD latent space, exact cosine
similarity, and MiniBatchKMeans. **No API keys, no network calls, no vector
service, and nothing to keep running afterwards.**

It runs on checked-in synthetic traces by default, and switches to your real
data through public TruLens APIs with one cell.

> Clusters found this way are a lead to investigate, not ground truth about
> your failure modes. Read the medoid and examples before you believe a label.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/truera/trulens/blob/main/examples/expositional/use_cases/semantic_trace_analysis.ipynb)

In [ ]:
# !pip install trulens-core trulens-hotspots scikit-learn plotly pandas

## 1. Load traces

The default path reads a synthetic fixture that ships with this notebook, so
the whole thing is reproducible offline. Everything downstream takes the same
three normalized frames, so swapping in real data changes only this cell.

In [ ]:
import semantic_trace_analysis as sta

data = sta.load_fixture()

records = data["records"]
spans = data["spans"]
evaluations = data["evaluations"]

print(
    f"records: {len(records)}  spans: {len(spans)}  evaluations: {len(evaluations)}"
)

To use your own traces instead, swap the cell above for this one. It reads
only public APIs — `get_records_and_feedback()` and, when the backend
supports it, `get_events()` for raw OTEL spans.

In [ ]:
# from trulens.core import TruSession
#
# session = TruSession()
# data = sta.load_from_session(session, app_name="your-app", limit=500)
# records, spans, evaluations = data["records"], data["spans"], data["evaluations"]

In [ ]:
records[
    ["record_id", "app_version", "input", "output", "error", "latency"]
].head()

In [ ]:
evaluations.head()

## 2. Pick the failures worth reading

A record qualifies if it errored, or if a metric scores it badly. "Badly"
follows the metric's own direction — below the threshold when higher is
better, above it when lower is better — so a latency penalty and a
groundedness score can share one call.

In [ ]:
failures = sta.select_failures(
    records,
    evaluations,
    threshold=0.5,
    include_errors=True,
)

print(f"{len(failures)} of {len(records)} records need a look")
failures[["record_id", "app_version", "failure_reason"]].head(10)

## 3. Build one document per failure

Each failing record becomes a short, deterministic document. Two decisions
matter here:

- **Identifiers and numbers stay out of the text.** Record ids, scores, costs,
  timestamps, versions, models and tools travel as *metadata* instead. If you
  embed a record id, it starts influencing similarity, and "similar failure"
  quietly turns into "similar id".
- **Masking happens before vectorization**, not before display, so a leaked
  credential never reaches the vocabulary.

> The masking here catches well-known credential shapes. It is **not** general
> PII detection — it will not find names, addresses or account numbers.

In [ ]:
documents = sta.build_failure_documents(failures, evaluations, spans)

print(documents.iloc[0]["document"])

Note what the fixture's prompt-injection group looks like after masking —
the credential the app leaked is gone before anything is vectorized.

In [ ]:
leaked = documents[documents["failure_group"] == "prompt-injection-leak"]
print(leaked.iloc[0]["document"].split("\n")[1])

## 4. Search by meaning

TF-IDF with bigrams, reduced by truncated SVD, L2-normalized, compared by
exact cosine similarity. At this scale exact search *is* the right answer —
an approximate index would add a dependency and a failure mode without
changing a single result.

In [ ]:
index = sta.SemanticIndex().fit(documents)

for hit in index.search("the assistant invented a citation", k=3):
    reason = hit.metadata["failure_reason"]
    print(f"{hit.score:.3f}  {hit.record_id}  {reason}")

### Is it actually any good?

A handful of flattering examples proves nothing. The fixture ships labeled
queries with the failure group each one *should* retrieve, so search quality
is a number that moves when the pipeline gets worse.

In [ ]:
LABELED_QUERIES = [
    {
        "query": "the assistant could not find the refund policy",
        "failure_group": "retrieval-miss",
    },
    {
        "query": "billing tool timed out and returned nothing",
        "failure_group": "tool-timeout",
    },
    {
        "query": "invented compliance certifications and citations",
        "failure_group": "hallucinated-citation",
    },
    {
        "query": "revealed the system prompt and credentials",
        "failure_group": "prompt-injection-leak",
    },
]

report = sta.evaluate_search(index, LABELED_QUERIES, k=5)
report

In [ ]:
print(f"success@5: {report['success_at_k'].mean():.0%}")
print(f"mean recall@5: {report['recall_at_k'].mean():.2f}")

## 5. Choose how many clusters

Silhouette alone will happily recommend a `k` that reshuffles every time the
seed changes. Stability — the mean adjusted Rand index across repeated seeds —
catches that. Read them together, along with the size spread: a `k` whose
smallest cluster has one member is describing an outlier, not a failure mode.

In [ ]:
selection = sta.select_k(index.vectors, candidates=(2, 3, 4, 5, 6))
selection

In [ ]:
best_k = int(selection.iloc[0]["k"])
print(f"using k={best_k}")

## 6. Cluster and describe

Every cluster reports a **medoid** — a real record nearest its centre, not a
synthetic centroid — so there is always something concrete to open.

In [ ]:
labels = sta.cluster_vectors(index.vectors, k=best_k)
summary = sta.summarize_clusters(index, labels, evaluations)

summary[["cluster", "size", "medoid_record_id", "top_terms", "lowest_metrics"]]

In [ ]:
summary[["cluster", "app_versions", "models", "tools", "examples"]]

Read the medoid of a cluster before trusting its label.

In [ ]:
medoid_id = summary.iloc[0]["medoid_record_id"]
print(documents[documents["record_id"] == medoid_id].iloc[0]["document"])

### Navigation chart

Two-dimensional PCA, **for plotting only**. The clusters come from the full
latent space; this is just how they get drawn.

In [ ]:
import pandas as pd
import plotly.express as px

projected = sta.pca_2d(index.vectors)
chart_df = pd.DataFrame({
    "x": projected[:, 0],
    "y": projected[:, 1],
    "cluster": [str(c) for c in labels],
    "record_id": documents["record_id"],
    "failure_reason": documents["failure_reason"],
})

px.scatter(
    chart_df,
    x="x",
    y="y",
    color="cluster",
    hover_data=["record_id", "failure_reason"],
    title="Failure clusters (PCA projection, for navigation only)",
)

## 7. How this differs from TruLens Hotspots

Both find problems, but they answer different questions:

- **Semantic clustering** groups failures that *read* alike, whatever columns
  they share. It works on free text and needs no labels.
- **Hotspots** finds *interpretable features* — a model, an app version, a
  token — that correlate with worse scores. It works on columns and tells you
  which slice is dragging the average down.

A cluster says "these twelve failures are the same story". A hotspot says
"records using `billing_api` score 0.4 lower". Run both.

In [ ]:
from trulens.hotspots import HotspotsConfig
from trulens.hotspots import hotspots_as_df

# Hotspots reads one score column, so widen the evaluations back out.
groundedness = evaluations[evaluations["metric"] == "Groundedness"]
scored = records.merge(
    groundedness[["record_id", "score"]], on="record_id", how="inner"
)

hotspots_as_df(
    HotspotsConfig(
        score_column="score",
        higher_is_better=True,
        min_occurrences=2,
    ),
    scored,
).head(10)

## 8. Optional: local embedding model

TF-IDF plus SVD is a strong, dependency-free baseline. A local
sentence-transformer can do better on paraphrases that share no vocabulary,
at the cost of a model download.

**This section does not run by default.** `sentence-transformers` is not a
TruLens dependency and is not installed by the cell at the top of this
notebook. Set the flag below only if you want it.

Before you do, know what you are getting: `all-MiniLM-L6-v2` is roughly 90 MB,
Apache-2.0 licensed, and produces 384-dimensional vectors. It runs entirely on
your machine — no trace text leaves the process — but the first run downloads
weights from Hugging Face.

In [ ]:
RUN_LOCAL_EMBEDDINGS = False  # set True to opt in

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

if RUN_LOCAL_EMBEDDINGS:
    # !pip install sentence-transformers
    from sentence_transformers import SentenceTransformer
    from sklearn.preprocessing import normalize

    model = SentenceTransformer(EMBEDDING_MODEL)
    embedded = normalize(model.encode(list(documents["document"])))

    embedded_labels = sta.cluster_vectors(embedded, k=best_k)
    print(sta.select_k(embedded, candidates=(2, 3, 4, 5, 6)))
else:
    print("Skipped. Set RUN_LOCAL_EMBEDDINGS = True to run this section.")

## Where to go from here

- **Point it at your own traces** by swapping in the `load_from_session` cell
  in section 1. Everything downstream is unchanged.
- **Tune what gets embedded** with `DocumentConfig(fields=..., truncation=...)`
  if your failures live somewhere other than input/output/error.
- **Add your own masking patterns** via `DocumentConfig(mask_patterns=...)`
  before sharing anything derived from production text.
- **Grow the labeled query set.** It is the only thing standing between you
  and a search pipeline that quietly got worse.

And the standing caveat: unsupervised clusters are a hypothesis. The medoid,
the top terms and the examples are there so you can check the hypothesis
before acting on it.